# 80 — Master Permit Registry

Builds a permit-aware national registry from documentary data and configured aliases.

In [ ]:
import os, io, re, json, hashlib, platform, sys
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import yaml

PROJECT_ROOT = Path.cwd().resolve().parents[0] if (Path.cwd().name == "notebooks") else Path.cwd().resolve()
DATA = PROJECT_ROOT / "data_sources"
OUTPUTS = PROJECT_ROOT / "outputs"
CONFIGS = PROJECT_ROOT / "configs"
OUTPUTS.mkdir(exist_ok=True)

def utcnow():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def write_json(p: Path, obj):
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")

def load_json(p: Path):
    return json.loads(p.read_text(encoding="utf-8"))

def load_yaml(p: Path):
    return yaml.safe_load(p.read_text(encoding="utf-8")) if p.exists() else {}

def safe_read_csv(path):
    p = Path(path)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    if not p.exists():
        return pd.DataFrame(), {"path": str(p), "status": "missing"}
    try:
        if p.stat().st_size == 0:
            return pd.DataFrame(), {"path": str(p), "status": "empty_file"}
        df = pd.read_csv(p)
        if df.empty:
            return pd.DataFrame(), {"path": str(p), "status": "no_rows", "sha256": sha256_file(p)}
        return df, {"path": str(p), "status": "ok", "sha256": sha256_file(p), "rows": int(len(df))}
    except Exception as e:
        return pd.DataFrame(), {"path": str(p), "status": f"error:{e}"}

def safe_read_parquet(path):
    p = Path(path)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    if not p.exists():
        return pd.DataFrame(), {"path": str(p), "status": "missing"}
    try:
        import pyarrow.parquet as pq
        df = pq.read_table(p).to_pandas()
        if df.empty:
            return pd.DataFrame(), {"path": str(p), "status": "no_rows", "sha256": sha256_file(p)}
        return df, {"path": str(p), "status": "ok", "sha256": sha256_file(p), "rows": int(len(df))}
    except Exception as e:
        return pd.DataFrame(), {"path": str(p), "status": f"error:{e}"}

def manifest_base(step_name: str, config_paths=None):
    return {
        "project": "AirQuality26",
        "step": step_name,
        "created_at_utc": utcnow(),
        "platform": {"python": sys.version, "platform": platform.platform()},
        "configs": {str(p): sha256_file(Path(p)) for p in (config_paths or []) if Path(p).exists()},
        "artifacts": [],
        "notes": []
    }

def add_artifact(manifest, p: Path):
    if p.exists():
        manifest["artifacts"].append({"path": str(p), "sha256": sha256_file(p)})

def slugify(s):
    return re.sub(r"[^a-z0-9]+", "_", str(s).lower()).strip("_")

def clean_facility(fn: str) -> str:
    s = re.sub(r"\.xlsm$|\.xlsx$|\.pdf$", "", str(fn), flags=re.I)
    s = re.sub(r"^[A-Z0-9]+\s+", "", s)
    s = re.sub(r"\s+Annual Report.*$", "", s, flags=re.I)
    s = re.sub(r"\s+Annual Performance Report.*$", "", s, flags=re.I)
    s = re.sub(r"\s+Report 2024$", "", s, flags=re.I)
    s = re.sub(r"\s+2024$", "", s)
    return s.strip()

phase80_cfg = load_yaml(CONFIGS / "phase80.yml")
permit_cfg = load_yaml(CONFIGS / "permit_registry.yml")
run_cfg = load_yaml(CONFIGS / "run.yml")

In [ ]:
step = "80_master_permit_registry"
out = OUTPUTS / step
out.mkdir(parents=True, exist_ok=True)

catalog, meta_catalog = safe_read_csv(DATA / "incinerator_report_catalog.csv")
if catalog.empty:
    raise FileNotFoundError("Need data_sources/incinerator_report_catalog.csv")

catalog["permit_id"] = catalog.get("permit_id", "").astype(str).str.strip()
catalog["facility_name"] = catalog.get("filename", "").map(clean_facility)
catalog["facility_key"] = catalog["facility_name"].map(slugify)
catalog["region_folder"] = catalog.get("region_folder", "")
catalog["operator"] = catalog.get("operator", "")
catalog["report_year"] = pd.to_numeric(catalog.get("report_year", catalog.get("year", np.nan)), errors="coerce")

manual = []
for site_id, spec in (permit_cfg.get("site_registry", {}) or {}).items():
    permits = spec.get("permit_ids", []) or []
    aliases = spec.get("aliases", []) or []
    for pid in permits:
        manual.append({"site_id": site_id, "permit_id": str(pid).strip(), "match_type": "manual_permit"})
    for alias in aliases:
        manual.append({"site_id": site_id, "facility_key_alias": slugify(alias), "match_type": "manual_alias"})
manual_df = pd.DataFrame(manual)

registry = catalog[["permit_id","facility_name","facility_key","region_folder","operator","report_year"]].drop_duplicates().copy()
registry["site_id"] = ""
if not manual_df.empty:
    if "permit_id" in manual_df.columns:
        pid_map = manual_df.dropna(subset=["permit_id"])[["permit_id","site_id"]].drop_duplicates()
        registry = registry.merge(pid_map, on="permit_id", how="left", suffixes=("","_pid"))
        registry["site_id"] = registry["site_id_pid"].fillna(registry["site_id"])
        registry = registry.drop(columns=[c for c in ["site_id_pid"] if c in registry.columns])
    alias_map = manual_df.dropna(subset=["facility_key_alias"])[["facility_key_alias","site_id"]].drop_duplicates()
    if not alias_map.empty:
        registry = registry.merge(alias_map, left_on="facility_key", right_on="facility_key_alias", how="left", suffixes=("","_alias"))
        registry["site_id"] = registry["site_id"].replace("", np.nan).fillna(registry["site_id_alias"]).fillna("")
        registry = registry.drop(columns=[c for c in ["facility_key_alias","site_id_alias"] if c in registry.columns])

registry["site_id"] = registry["site_id"].replace("", np.nan)
registry["linked_to_repo_site"] = registry["site_id"].notna()

registry.to_csv(out / "master_permit_registry.csv", index=False)
write_json(out / "master_permit_registry_summary.json", {
    "rows": int(len(registry)),
    "unique_permits": int(registry["permit_id"].nunique()),
    "linked_to_repo_site_rows": int(registry["linked_to_repo_site"].sum()),
})

manifest = manifest_base(step, [CONFIGS / "permit_registry.yml"])
manifest["inputs"] = {"catalog": meta_catalog}
add_artifact(manifest, out / "master_permit_registry.csv")
add_artifact(manifest, out / "master_permit_registry_summary.json")
write_json(out / "manifest.json", manifest)
display(registry.head(20))